# 3. Probabilidades out-of-fold com `cross_val_predict`
Este notebook mostra como cada observação do treino recebe uma probabilidade de um modelo que não a usou no ajuste. O OOF é uma estimativa diagnóstica do treino; neste projeto o threshold é escolhido no conjunto separado de validação, e o teste fica para a avaliação final.


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, fbeta_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_predict
from breast_cancer.config import DATA_PATH, THRESHOLD_BETA
from breast_cancer.data import load_data
from breast_cancer.evaluation import evaluate_model
from breast_cancer.validation import select_threshold
from breast_cancer.models import make_xgboost
from breast_cancer.preprocessing import prepare_data

(X_train, X_validation, X_test,
 y_train, y_validation, y_test) = prepare_data(load_data(DATA_PATH))
print(f'Treino: {X_train.shape}; validação: {X_validation.shape}; teste: {X_test.shape}')


Treino: (341, 30); validação: (114, 30); teste: (114, 30)


## Gerar as previsões OOF no treino

Com cinco folds, o modelo é ajustado em quatro partes do treino e prediz a parte restante. Cada linha recebe uma previsão de um modelo que não a treinou. `[:, 1]` seleciona a probabilidade da classe maligna (`1`).


In [2]:
model = make_xgboost()
y_proba_oof = cross_val_predict(
    model, X_train, y_train, cv=5, method='predict_proba', n_jobs=1
)[:, 1]
print(f'Probabilidades OOF: {len(y_proba_oof)} (uma por linha de treino)')
print(f'AUC-ROC OOF: {roc_auc_score(y_train, y_proba_oof):.4f}')


Probabilidades OOF: 341 (uma por linha de treino)
AUC-ROC OOF: 0.9874


## Comparar com probabilidades in-sample

Ajustar e prever sobre as mesmas linhas tende a produzir uma estimativa otimista. O OOF avalia cada observação com um modelo que não a viu durante o ajuste.


In [3]:
model.fit(X_train, y_train)
y_proba_in_sample = model.predict_proba(X_train)[:, 1]
comparison = pd.DataFrame(
    {'AUC-ROC': [roc_auc_score(y_train, y_proba_oof),
                 roc_auc_score(y_train, y_proba_in_sample)]},
    index=['OOF (treino)', 'In-sample (treino visto pelo modelo)'],
)
comparison


,AUC-ROC
OOF (treino),0.987379
In-sample (treino visto pelo modelo),0.998970


## Threshold escolhido na validação

Depois de ajustar o XGBoost no treino, escolhemos o threshold que maximiza F2 usando somente o conjunto de validação. O teste não participa da escolha.


In [4]:
validation_probabilities = model.predict_proba(X_validation)[:, 1]
threshold = select_threshold(y_validation, validation_probabilities, beta=THRESHOLD_BETA)
validation_metrics = evaluate_model(y_validation, validation_probabilities, threshold)
oof_metrics = evaluate_model(y_train, y_proba_oof, threshold)
test_probabilities = model.predict_proba(X_test)[:, 1]
test_metrics = evaluate_model(y_test, test_probabilities, threshold)

print(f'Threshold escolhido na validação (F2, beta={THRESHOLD_BETA:g}): {threshold:.4f}')
print(f"Validação | precision: {precision_score(y_validation, validation_metrics['predicted_labels']):.3f}; recall: {recall_score(y_validation, validation_metrics['predicted_labels']):.3f}; F1: {f1_score(y_validation, validation_metrics['predicted_labels']):.3f}; F2: {fbeta_score(y_validation, validation_metrics['predicted_labels'], beta=THRESHOLD_BETA):.3f}")
print(f"OOF no threshold da validação | recall: {recall_score(y_train, oof_metrics['predicted_labels']):.3f}; F1: {f1_score(y_train, oof_metrics['predicted_labels']):.3f}")
print(f"Teste intocado | AUC-ROC: {test_metrics['auc_roc']:.4f}; accuracy: {test_metrics['accuracy']:.4f}")
print(test_metrics['classification_report'])


Threshold escolhido na validação (F2, beta=2): 0.2247
Validação | precision: 0.977; recall: 1.000; F1: 0.989; F2: 0.995
OOF no threshold da validação | recall: 0.961; F1: 0.917
Teste intocado | AUC-ROC: 0.9947; accuracy: 0.9737
              precision    recall  f1-score   support

     Benigno      0.986     0.972     0.979        72
     Maligno      0.953     0.976     0.965        42

    accuracy                          0.974       114
   macro avg      0.970     0.974     0.972       114
weighted avg      0.974     0.974     0.974       114



## Como interpretar

- OOF gera estimativas para o treino; ele não é usado para escolher o threshold neste fluxo.
- O threshold é ajustado na validação, em dados separados do treino.
- A avaliação final é feita uma vez no teste, que não participou do treino, do CV ou da escolha do threshold.
- A performance de validação no threshold selecionado pode ser otimista pela própria seleção; use o teste para avaliar a regra completa.
